### Predicting Storm Occurrence Using Random Forest Classifier

In this section, we aim to build a classification model to predict **whether or not a tropical storm will occur** using a **Random Forest Classifier** and based on based on climate-related variables such as CO₂ emissions, monthly global surface temperatures, and date information (year, month, day). This binary prediction forms the first part of a two-step pipeline, where the second part predicts storm intensity if a storm is expected.

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

### Load and Explore the Dataset
We begin by loading the merged dataset completed_dataset_for_IS_project_25.csv, which includes data from historical storm records, global temperatures, and CO₂ emissions. A quick preview helps ensure the dataset loads correctly and has all necessary columns.

In [2]:
# Load dataset
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")
print("Initial dataset shape:", data.shape)

# Engineer the 'storm_occured' column based on intensity label
# Label as 1 (storm occurred) if intensity > 1, else 0
data['storm_occured'] = (data['Storm Intensity Label'] > 1).astype(int)

# Preview the result
print(data['storm_occured'].value_counts())
data.head()

Initial dataset shape: (52656, 26)
storm_occured
1    38639
0    14017
Name: count, dtype: int64


,Year,MONTH,DAY,LAT,LONG,WIND_KTS,PRESSURE,CAT,Shape_Leng,Country,...,Jul,Aug,Sep,Oct,Nov,Dec,Storm Intensity,Storm Intensity Label,Wind Speed Squared,storm_occured
0,1880,8,11,23.0,-91.9,70,0,H1,0.806226,Mexico,...,-0.18,-0.11,-0.15,-0.24,-0.22,-0.18,56.43582,3,4900,1
1,1880,8,11,23.4,-92.6,80,0,H1,0.761577,Mexico,...,-0.18,-0.11,-0.15,-0.24,-0.22,-0.18,60.92616,3,6400,1
2,1880,8,11,23.7,-93.3,80,0,H1,0.583095,Mexico,...,-0.18,-0.11,-0.15,-0.24,-0.22,-0.18,46.64760,3,6400,1
3,1880,8,12,24.0,-93.8,90,0,H2,0.670820,Mexico,...,-0.18,-0.11,-0.15,-0.24,-0.22,-0.18,60.37380,4,8100,1
4,1880,9,6,23.9,-88.6,40,0,TS,0.360555,Mexico,...,-0.18,-0.11,-0.15,-0.24,-0.22,-0.18,14.42220,2,1600,1


### Define Target and Feature Variables
To enable storm occurrence classification, we:

Define the binary target variable storm_occured, which indicates whether a storm was present (1) or not (0).

Select feature columns that include:
Temporal variables (Year, MONTH, DAY)
Environmental variables (CO2 emission (Tons))
Monthly surface temperature anomalies (Jan to Dec)

We drop features related to storm severity (Storm Intensity, Storm Intensity Label, WIND_KTS, etc.) to prevent data leakage during occurrence prediction.

In [3]:
# Drop irrelevant columns & set up features
features = [
    "Year", "MONTH", "DAY", "CO2 emission (Tons)",
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]
target = "storm_occured"

X = data[features]
y = data[target]

### Split the Dataset (Train/Val/Test)
To validate model performance robustly, the dataset is split into:

Training Set (70%)

Validation Set (15%)

Test Set (15%)

We use stratified sampling to ensure that both classes (storm/no storm) are equally represented across splits, avoiding class imbalance issues that might skew the model.

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

Train size: 36859
Validation size: 7898
Test size: 7899


### Train the Random Forest Classifier
We train a Random Forest Classifier with the following settings:

n_estimators=100: the number of decision trees in the forest
max_depth=8: limits tree depth to reduce overfitting
random_state=5: for reproducibility

This ensemble method is chosen for its robustness, ability to handle nonlinear relationships, and interpretability.

In [6]:
# Initialize model with basic hyperparameters
model_rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=8)
model_rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=8, random_state=42)

### Evaluate the Model (Test Set)
We evaluate our trained model on the unseen test set, using the following metrics:

Accuracy: Overall correctness
Precision: Focus on minimizing false positives
F1-Score: Balance between precision and recall
Latency: How quickly the model generates predictions

This helps us assess real-world performance in both correctness and responsiveness.

In [7]:
# Test Set Evaluation
start_time = time.time()
y_pred_test = model_rf.predict(X_test)
latency_test = time.time() - start_time

print("📊 Evaluation on Test Set")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_test):.4f}")
print(f"Latency: {latency_test:.4f} seconds")

📊 Evaluation on Test Set
Accuracy: 0.7472
Precision: 0.7471
F1 Score: 0.8519
Latency: 0.1077 seconds
